# Analisi delle Comunità e dei Contenuti di Threads
Questo notebook riproduce i calcoli eseguiti per confrontare le query **AI**, **ChatGPT** e **ML** su Threads.

## 1. Analisi della Comunità Utenti
Calcolo dei seguenti indicatori per ciascuna query:
- Numero di utenti unici
- Numero di follower unici
- Numero di relazioni follower totali
- Numero medio di follower per utente

In [1]:
from pathlib import Path
DATA_DIR = Path.cwd()
DATA_DIR = DATA_DIR.parent.parent / 'data' / 'interim'
DATA_DIR

WindowsPath('c:/Users/pasqu/Desktop/progettoasnm/Code/data_extraction/data/interim')

In [2]:
import pandas as pd

# Caricamento dati follower
df_ai = pd.read_csv(DATA_DIR / 'ai_all_followers.csv')
df_chatgpt = pd.read_csv(DATA_DIR /'chatgpt_all_followers.csv')
df_ml = pd.read_csv(DATA_DIR / 'ml_all_followers.csv')

# Funzione per calcolare metriche
def compute_follow_metrics(df):
    num_users = df['thread_user_pk'].nunique()
    num_followers = df['thread_follower_pk'].nunique()
    total_relations = len(df)
    avg_followers_per_user = total_relations / num_users if num_users else 0
    return num_users, num_followers, total_relations, avg_followers_per_user

metrics = {
    'AI': compute_follow_metrics(df_ai),
    'ChatGPT': compute_follow_metrics(df_chatgpt),
    'ML': compute_follow_metrics(df_ml)
}

# Costruzione DataFrame risultati
results_community = pd.DataFrame.from_dict(
    metrics, orient='index',
    columns=['Utenti unici', 'Follower unici', 'Relazioni totali', 'Follower per utente (medio)']
)
results_community

,Utenti unici,Follower unici,Relazioni totali,Follower per utente (medio)
AI,3060,327953,751464,245.576471
ChatGPT,1084,10354,13486,12.440959
ML,3551,345261,458631,129.155449


## 2. Analisi dei Contenuti dei Post

- Unione dei dataset `total_post1.csv`, `total_post2.csv`, `total_post3.csv`.
- Calcolo di post totali, utenti poster, post per utente medio, like medi, repost medi, reshare medi, lunghezza media.

In [3]:
# Caricamento dati post
dfs_posts = [pd.read_csv(DATA_DIR.parent / 'processed' / 'post_data' / f'total_post{i}.csv',low_memory=False) for i in (1, 2, 3)]
df_posts = pd.concat(dfs_posts, ignore_index=True)

# Insiemi di utenti per query
users_ai = set(df_ai['thread_user_pk'])
users_chatgpt = set(df_chatgpt['thread_user_pk'])
users_ml = set(df_ml['thread_user_pk'])

def compute_post_metrics(df, user_set):
    subset = df[df['thread_user_pk'].isin(user_set)].copy()
    num_posts = len(subset)
    num_poster = subset['thread_user_pk'].nunique()
    avg_posts_per_user = num_posts / num_poster if num_poster else 0
    avg_likes = subset['like_count'].mean()
    avg_reposts = subset['repost_count'].mean() if 'repost_count' in subset else 0
    avg_reshares = subset['reshare_count'].mean() if 'reshare_count' in subset else 0
    # Convert caption_text to string and handle missing values
    avg_length = subset['caption_text'].fillna('').astype(str).str.split().map(len).mean()
    return num_posts, num_poster, avg_posts_per_user, avg_likes, avg_reposts, avg_reshares, avg_length

post_metrics = {
    'AI': compute_post_metrics(df_posts, users_ai),
    'ChatGPT': compute_post_metrics(df_posts, users_chatgpt),
    'ML': compute_post_metrics(df_posts, users_ml)
}

columns = ['Post totali', 'Utenti poster', 'Post per utente', 'Like medi', 'Repost medi', 'Reshare medi', 'Lunghezza media (parole)']
results_posts = pd.DataFrame.from_dict(post_metrics, orient='index', columns=columns)
results_posts

,Post totali,Utenti poster,Post per utente,Like medi,Repost medi,Reshare medi,Lunghezza media (parole)
AI,38465,2036,18.892436,64.526817,2.374210,1.925153,17.156714
ChatGPT,17207,930,18.502151,90.966816,5.155053,2.702156,21.973964
ML,24184,1001,24.159840,21.659858,0.929582,0.484287,24.381657


## 3. Sovrapposizione e Distinzione

Calcolo del numero di utenti comuni tra le query.

In [4]:
# Calcolo delle intersezioni tra set di utenti
overlaps = {
    'AI & ML': len(users_ai & users_ml),
    'AI & ChatGPT': len(users_ai & users_chatgpt),
    'ChatGPT & ML': len(users_chatgpt & users_ml),
    'Tutte e tre': len(users_ai & users_chatgpt & users_ml)
}
results_overlap = pd.DataFrame.from_dict(
    overlaps, orient='index', columns=['Utenti comuni']
)
results_overlap

,Utenti comuni
AI & ML,429
AI & ChatGPT,3
ChatGPT & ML,2
Tutte e tre,0


## 4. Considerazioni Generali

Tendenze emergenti:

- La forte sovrapposizione AI–ML indica comunità parallele ma convergenti, probabilmente focalizzate su argomenti di data science e modelli.

- L’engagement elevatissimo sui post “AI” suggerisce contenuti dal forte appeal virale (demo, rilasci di modelli, discussioni su nuove tecnologie).

- La nicchia “ChatGPT” appare ancora in fase esplorativa, con pochi utenti “core” che postano frequentemente ma faticano a coinvolgere un pubblico più ampio.

Insight aggiuntivi:

- I temi estratti (“all”, “what”, “just”…) sono troppo generici: serve un vocabolario mirato (es. “model”, “training”, “parameters”) per isolare i topic tecnici.

- Non essendo disponibili dati temporali sui follower, non è possibile inferire una crescita passata o tassi di acquisizione; occorrerebbero snapshot multipli.

